# 04.2 · kv-transfer (Spec-2) — data pipeline demo

`SRC.task != DST.task`, `SRC.progress == DST.progress`. Two distinct tasks that share a **common progress** (a prerequisite sub-computation) are clustered together; both prefill the same progress code. Transplant the SRC **task-description key+value** onto DST (GQA-corrected headwise routing) → does DST implement SRC's task?

Also demonstrates **cross-language** transfer: cache the task in Python, trigger it in C++.

In [1]:
%load_ext autoreload
%autoreload 2
import os, sys

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "2, 3, 4")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

sys.path.append("../..")  # repo root, so `src` is importable
import torch, numpy as np
import src.diff.functional as P
from src.diff.functional import *          # helpers into scope
import src.diff.data as D                   # dataset generators

P.setup("Qwen/Qwen2.5-Coder-32B-Instruct") # loads model + head masks (once)
mt = P.mt
GOAL_HEADS, BOUNDARY_HEADS, UNION_HEADS, GROUP = P.GOAL_HEADS, P.BOUNDARY_HEADS, P.UNION_HEADS, P.GROUP
PATCH_HEADS = GOAL_HEADS                    # heads carrying the transplant (GOAL | UNION | ...)
from rich.panel import Panel
from rich.text import Text
from rich.markup import escape

def pivot_of(prompt, neg_shift=-1):
    return prepare_input(prompts=prompt, tokenizer=mt)["input_ids"].shape[1] + neg_shift

def show(prompt, title, emph_span=None, piv=-1):
    kw = dict(patch_pos=[piv])
    if emph_span is not None:
        kw["emphasize_tokens"] = range(*emph_span)
    rich_print(Panel(Text.from_markup(exact_prompt(prompt, **kw)),
                     title=f"[bold]{title}[/bold]", title_align="left",
                     border_style="grey42", padding=(1, 1)))

2026-09-02 13:18:36 src.models WARNING  Qwen/Qwen2.5-Coder-32B-Instruct not found in /disk/u/arnab/Codes/Models
If not found in cache, model will be downloaded from HuggingFace to cache directory: /disk/u/models


Loading weights:   0%|          | 0/771 [00:00<?, ?it/s]

Qwen/Qwen2.5-Coder-32B-Instruct
n_heads=40  n_kv_heads=8  group=5  head_dim=128  n_layer=64
GOAL_HEADS     : 122  (layers 28–55)
BOUNDARY_HEADS : 140  (layers 11–62)
UNION_HEADS    : 228


## Inspect the generated clusters / setups

In [2]:
for dt, cl in D.KVCLUSTERS.items():
    print(f"{dt:<10} cluster {cl.id!r}: common_progress={cl.common_progress.result_name!r} · "
          f"tasks={[t.id for t in cl.tasks]}")
print()
setups = D.kv_transfer_setups("list[str]", presentation="description")
for s in setups:
    print(f"  src_task={s.src_task:<22} dst_task={s.dst_task:<22} pivot={s.pivot}")

list[str]  cluster 'sorted_by_len': common_progress='sorted_words' · tasks=['shortest_vowel_start', 'longest_palindrome', 'longest_len', 'shortest_len']
list[int]  cluster 'sorted_asc': common_progress='sorted_nums' · tasks=['median_is_even', 'range_span', 'second_largest', 'smallest_positive']
int        cluster 'digits': common_progress='digits' · tasks=['digit_sum', 'max_digit', 'count_even_digits', 'all_distinct_digits']
str        cluster 'chars_sorted': common_progress='sorted_chars' · tasks=['first_char_vowel', 'last_char', 'has_repeat', 'distinct_chars']

  src_task=shortest_vowel_start   dst_task=longest_palindrome     pivot=return
  src_task=longest_palindrome     dst_task=longest_len            pivot=return
  src_task=longest_len            dst_task=shortest_len           pivot=return
  src_task=shortest_len           dst_task=shortest_vowel_start   pivot=return


## Render one setup — SRC/DST prompts with the transplanted task span marked

In [3]:
inst = D.kv_transfer_setups("list[str]", presentation="description",
                            src_lang="python", dst_lang="python")[0]
m = D.materialize(inst)
src_prompt, dst_prompt = m["src_prompt"], m["dst_prompt"]
SRC_SPAN = token_span(src_prompt, m["src_task_text"])   # NL task description = transplant span
DST_SPAN = token_span(dst_prompt, m["dst_task_text"])
piv_src, piv_dst = pivot_of(src_prompt), pivot_of(dst_prompt)
print(f"SRC task = {m['src_task_text']!r}\nDST task = {m['dst_task_text']!r}")
show(src_prompt, f"SRC · task span {SRC_SPAN} (yellow) · pivot@{piv_src}", emph_span=SRC_SPAN, piv=-1)
show(dst_prompt, f"DST · task span {DST_SPAN} (yellow) · pivot@{piv_dst}", emph_span=DST_SPAN, piv=-1)

SRC task = 'returns true if the shortest word starts with a vowel'
DST task = 'returns true if the longest word is a palindrome'


╭─ SRC · task span (63, 73) (yellow) · pivot@101 ─────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ returns true if the shortest word starts with a vowel\n                                                         │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     sorted_words = sorted(words, key=len)\n                                                                     │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ DST · task span (63, 72) (yellow) · pivot@100 ─────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│ <|im_start|>system\n                                                                                            │
│ You are a helpful assistant proficient in programming. Always wrap your code inside <code></code> blocks. Just  │
│ the code, no need to comment or explain.<|im_end|>\n                                                            │
│ <|im_start|>user\n                                                                                              │
│ Write a Python program that implements a function `func` which, given a list of strings `words`, computes:\n    │
│ returns true if the longest word is a palindrome\n                                                              │
│ <|im_end|>\n                                                                                                    │
│ <|im_start|>assistant\n                                                                                         │
│ <code>\n                                                                                                        │
│ def func(words: list[str]):\n                                                                                   │
│     sorted_words = sorted(words, key=len)\n                                                                     │
│     return                                                                                                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

## Intervention — transplant SRC task-description key+value onto DST

Right-edge align the two task spans (Spec-2 convention), cache SRC key+value over `PATCH_HEADS`, and run the destination under `headwise_kv_patch` (only `PATCH_HEADS` read the source span — GQA bleed corrected). Expect patched DST to implement SRC's task.

In [4]:
# ── intervention: transplant SRC task-description k+v onto DST — grouped per run ──
from src.attention import get_attention_matrices
K = min(SRC_SPAN[1] - SRC_SPAN[0], DST_SPAN[1] - DST_SPAN[0])
SRC_POS = list(range(SRC_SPAN[1] - K, SRC_SPAN[1]))   # right-edge aligned (Spec-2 convention)
DST_POS = list(range(DST_SPAN[1] - K, DST_SPAN[1]))

kvheads = heads_per_layer(PATCH_HEADS, "k", layers=set(_layers_of(PATCH_HEADS)))
k_cache = cache_src_proj(src_prompt, SRC_POS, sorted(kvheads), "k")
v_cache = cache_src_proj(src_prompt, SRC_POS, sorted(kvheads), "v")

src_tok = prepare_input(prompts=src_prompt, tokenizer=mt)
dst_tok = prepare_input(prompts=dst_prompt, tokenizer=mt)
_iface  = make_headwise_interface(mt, PATCH_HEADS, k_cache, v_cache, DST_POS)   # GQA-corrected

# clean baseline attn over the DST task-description span; query = pivot (last token)
TASK_POS  = list(range(*DST_SPAN))
task_toks = [mt.tokenizer.decode([int(dst_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_i  = get_attention_matrices(dst_tok, mt, value_weighted=True)
def _cattn(h): return pivot_attn(_clean_i, list(h), TASK_POS, piv_dst)

# ── source_run — clean SRC (its own task); generation + goalscope ──
rich_print(f"[red1]source_run · clean SRC ({m['src_task_text']!r}):[/red1] {escape(repr(greedy(src_tok, max_new=64)))}")
goalscope_readout("   ↳ goalscope · source_run", src_prompt, piv_src, heads=PATCH_HEADS)

# ── destination_run — clean DST; generation + goalscope + ATTN_VIS baseline ──
rich_print(f"[sea_green3]destination_run · clean DST ({m['dst_task_text']!r}):[/sea_green3] {escape(repr(greedy(dst_tok, max_new=64)))}")
goalscope_readout("   ↳ goalscope · destination_run", dst_prompt, piv_dst, heads=PATCH_HEADS)
show_delta_tokens(task_toks, _cattn(PATCH_HEADS), "ATTN_VIS · pivot→TASK (clean · PATCH_HEADS)")
show_delta_tokens(task_toks, _cattn(GOAL_HEADS),  "ATTN_VIS · pivot→TASK (clean · GOAL_HEADS)")

# ── patched_run — SRC task k+v transplanted (expect SRC's task); gen + goalscope + ATTN_DIFF ──
with headwise_kv_patch(mt, PATCH_HEADS, k_cache, v_cache, DST_POS):
    _patched = greedy(dst_tok, max_new=64)
rich_print(f"[salmon1]patched_run · DST (expect {m['src_task_text']!r}):[/salmon1] {escape(repr(_patched))}")
goalscope_readout("   ↳ goalscope · patched_run", dst_prompt, piv_dst, heads=PATCH_HEADS, iface=_iface)
_patched_i = get_attention_matrices(dst_tok, mt, value_weighted=True, patch_interface=_iface)
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(PATCH_HEADS), TASK_POS, piv_dst) - _cattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(GOAL_HEADS), TASK_POS, piv_dst),
                  "ATTN_VIS · patched (GOAL_HEADS)")

NameError: name '_layers_of' is not defined

## Cross-language — cache the task in Python, trigger it in a C++ destination

In [ ]:
# ── cross-language: cache the task in Python, trigger it in a C++ destination — grouped per run ──
from src.attention import get_attention_matrices
xinst = D.kv_transfer_setups("list[str]", presentation="description",
                             src_lang="python", dst_lang="cpp")[0]
xm = D.materialize(xinst)
xsrc, xdst = xm["src_prompt"], xm["dst_prompt"]
XS, XD = token_span(xsrc, xm["src_task_text"]), token_span(xdst, xm["dst_task_text"])
xpiv_src, xpiv_dst = pivot_of(xsrc), pivot_of(xdst)
show(xsrc, f"SRC (python) · task span {XS}", emph_span=XS, piv=-1)
show(xdst, f"DST (c++)    · task span {XD}", emph_span=XD, piv=-1)

XK  = min(XS[1] - XS[0], XD[1] - XD[0])
XSP = list(range(XS[1] - XK, XS[1])); XDP = list(range(XD[1] - XK, XD[1]))
xkvh = heads_per_layer(PATCH_HEADS, "k", layers=set(_layers_of(PATCH_HEADS)))
xk = cache_src_proj(xsrc, XSP, sorted(xkvh), "k")
xv = cache_src_proj(xsrc, XSP, sorted(xkvh), "v")
xsrc_tok = prepare_input(prompts=xsrc, tokenizer=mt)
xdst_tok = prepare_input(prompts=xdst, tokenizer=mt)
_xiface  = make_headwise_interface(mt, PATCH_HEADS, xk, xv, XDP)

TASK_POS  = list(range(*XD))
task_toks = [mt.tokenizer.decode([int(xdst_tok["input_ids"][0][j])]) for j in TASK_POS]
_clean_i  = get_attention_matrices(xdst_tok, mt, value_weighted=True)
def _cattn(h): return pivot_attn(_clean_i, list(h), TASK_POS, xpiv_dst)

# ── source_run (python) ──
rich_print(f"[red1]source_run · clean SRC/python ({xm['src_task_text']!r}):[/red1] {escape(repr(greedy(xsrc_tok, max_new=64)))}")
goalscope_readout("   ↳ goalscope · source_run", xsrc, xpiv_src, heads=PATCH_HEADS)

# ── destination_run (c++); ATTN_VIS baseline ──
rich_print(f"[sea_green3]destination_run · clean DST/c++ ({xm['dst_task_text']!r}):[/sea_green3] {escape(repr(greedy(xdst_tok, max_new=64)))}")
goalscope_readout("   ↳ goalscope · destination_run", xdst, xpiv_dst, heads=PATCH_HEADS)
show_delta_tokens(task_toks, _cattn(PATCH_HEADS), "ATTN_VIS · pivot→TASK (clean c++ DST · PATCH_HEADS)")
show_delta_tokens(task_toks, _cattn(GOAL_HEADS),  "ATTN_VIS · pivot→TASK (clean c++ DST · GOAL_HEADS)")

# ── patched_run — python task k+v -> c++ DST (expect the PYTHON src task) ──
with headwise_kv_patch(mt, PATCH_HEADS, xk, xv, XDP):
    xpatched = greedy(xdst_tok, max_new=64)
rich_print(f"[salmon1]patched_run · DST/c++ (expect {xm['src_task_text']!r}):[/salmon1] {escape(repr(xpatched))}")
goalscope_readout("   ↳ goalscope · patched_run", xdst, xpiv_dst, heads=PATCH_HEADS, iface=_xiface)
_patched_i = get_attention_matrices(xdst_tok, mt, value_weighted=True, patch_interface=_xiface)
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(PATCH_HEADS), TASK_POS, xpiv_dst) - _cattn(PATCH_HEADS),
                  "ATTN_DIFF · patched−clean (PATCH_HEADS)")
show_delta_tokens(task_toks, pivot_attn(_patched_i, list(GOAL_HEADS), TASK_POS, xpiv_dst),
                  "ATTN_VIS · patched (GOAL_HEADS)")